# Part 1.

In [1]:
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from tqdm import tqdm

## 1. Prepare Dataset

In [ ]:
# Load text pairs and split them into training and validation sets
def split_train_test(path, valid_lines):
    with Path(path).open("r", encoding="utf-8", errors="replace") as f:
        pairs = [tuple(line.rstrip("\n").lstrip("\ufeff").split("\t", 1)) for line in f]

    split = len(pairs) - valid_lines

    def expand(ps):
        X, y = [], []
        for l, r in ps:
            X += [l, r]
            y += [1, 0]
        return X, y

    X_train, y_train = expand(pairs[:split])
    idx = np.random.RandomState(seed=42).permutation(len(X_train))
    X_train, y_train = [X_train[i] for i in idx], [y_train[i] for i in idx]

    X_valid, y_valid = expand(pairs[split:])
    return X_train, X_valid, y_train, y_valid

X_train_texts, X_valid_texts, y_train, y_valid = split_train_test("/content/train.txt", valid_lines=100_000)
print(len(X_train_texts), len(X_valid_texts))

1800000 200000


## 2. Train Model

In [ ]:
# Convert the texts into character-level TF-IDF feature matrices
tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(1, 4),
    min_df=2,
    lowercase=False,
    dtype=np.float32
)
Xtr = tfidf.fit_transform(tqdm(X_train_texts, unit="doc"))
Xva = tfidf.transform(tqdm(X_valid_texts, unit="doc"))

100%|██████████| 200000/200000 [00:45<00:00, 4430.85doc/s]


In [ ]:
# Train a logistic regression and evaluate it by choosing the higher-scoring text in each validation pair
clf = LogisticRegression(
    solver="saga",
    penalty="l2",
    max_iter=4000,
    C=2,
    random_state=42,
    verbose=1
)
clf.fit(Xtr, y_train)

probs = clf.predict_proba(Xva)[:, 1]
pair_preds = []
for i in range(0, len(probs), 2):
    if probs[i] >= probs[i+1]:
        pair_preds.extend([1, 0])
    else:
        pair_preds.extend([0, 1])

acc = accuracy_score(y_valid, pair_preds)
print(f"{acc:.4f} ({(np.array(pair_preds) == np.array(y_valid)).sum()}/{len(y_valid)})")

convergence after 19 epochs took 376 seconds


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:  6.3min finished


0.8850 (176994/200000)


## 3. Output File

In [5]:
test_pairs = []
with Path("/content/test.rand.txt").open("r", encoding="utf-8", errors="replace") as f:
    for raw in f:
        line = raw.rstrip("\n").lstrip("\ufeff")
        a, b = line.split("\t", 1)
        test_pairs.append((a, b))

test_texts = []
for a, b in test_pairs:
    test_texts.append(a)
    test_texts.append(b)

Xte = tfidf.transform(tqdm(test_texts, unit="doc"))
probs = clf.predict_proba(Xte)[:, 1]

with Path("part1.txt").open("w", encoding="utf-8", newline="\n") as out:
    for i in range(0, len(probs), 2):
        pa, pb = probs[i], probs[i+1]
        out.write("A\n" if pa >= pb else "B\n")

100%|██████████| 200000/200000 [00:41<00:00, 4802.79doc/s]


# Part 2.

## 1. Corrupt Sentences

In [6]:
import random
import re

In [ ]:
# Read the left text from each training pair, replacing any remaining tabs with spaces
def read_pairs_train(path):
    for raw in Path(path).open("r", encoding="utf-8", errors="replace"):
        line = raw.rstrip("\n").lstrip("\ufeff")
        left = line.split("\t", 1)[0]
        yield left.replace("\t", " ")

In [ ]:
random.seed(42)

def corrupt_sentence(s):
    s = re.sub(r"\s+", " ", s).strip()
    words = s.split(" ")

    # Swap common function words 
    def frequent_word_swap(ws):
        before = " ".join(ws)
        swaps = {
            "is": "are",
            "are": "is",
            "was": "were",
            "were": "was",
            "do": "does",
            "does": "do",
            "have": "has",
            "has": "have",
            "what": "when",
            "when": "how",
            "how": "who",
            "who": "where",
            "where": "whose",
            "whose": "which",
            "which": "why",
            "why": "what",
        }

        pattern = r"\b(" + "|".join(swaps.keys()) + r")\b"

        def repl(match):
            word = match.group(0)
            repl_word = swaps[word.lower()]
            if word[0].isupper():
                repl_word = repl_word.capitalize()
            return repl_word

        after, count = re.subn(pattern, repl, before, count=1, flags=re.IGNORECASE)

        if count > 0:
            ws[:] = after.split(" ")
            return True
        return False

    # Drop the final 's' or 'es' from a token if it exists
    def drop_final_s_es(ws):
        candidates = [i for i, w in enumerate(ws) if len(w) > 1 and w.endswith('s')]
        if not candidates:
            return False

        idx = random.choice(candidates)
        w = ws[idx]

        if len(w) > 2 and w[-2] == 'e':
            ws[idx] = w[:-2]
        else:
            ws[idx] = w[:-1]

        return True

    adj_suffixes = ("able","ible","less","like","ous","ive","ful","ary","ish","ian","ese","al","ly")

    # Replace the suffix
    def swap_suffix(ws, target_suffixes):
        suffixes = sorted(set(target_suffixes), key=len, reverse=True)

        for idx, w in enumerate(ws):
            m = re.match(r'^([A-Za-z]+)([^A-Za-z]*)$', w)
            if not m:
                continue
            stem, trail = m.groups()
            low = stem.lower()

            matched = None
            for suf in suffixes:
                if low.endswith(suf) and len(stem) > len(suf):
                    matched = suf
                    break
            if not matched:
                continue

            choices = [s for s in suffixes if s != matched]
            if not choices:
                return False
            new_suf = random.choice(choices)
            new_stem = stem[:-len(matched)] + new_suf

            if stem.istitle():
                new_stem = new_stem[0].upper() + new_stem[1:]

            ws[idx] = new_stem + trail
            return True

        return False

    # Swap two different words that end with one of the target suffixes
    def swap_suffix_words(ws, target_suffixes):
        matches = [(i, w) for i, w in enumerate(ws) if w.lower().endswith(target_suffixes)]

        if len(matches) < 2:
            return False

        unique_pairs = [(i, w) for i, w in matches if w.lower() != matches[0][1].lower()]
        if not unique_pairs:
            return False

        i1, w1 = random.choice(matches)
        other_matches = [(i, w) for i, w in matches if i != i1 and w.lower() != w1.lower()]
        if not other_matches:
            return False
        i2, w2 = random.choice(other_matches)
        ws[i1], ws[i2] = ws[i2], ws[i1]
        return True

    # Change the first eligible plural ending between "s" and "es"
    def plural_swap(ws):
      for i, w in enumerate(ws):
          lw = w.lower()
          if lw.endswith("es") and not lw.endswith("ses"):
              new_w = w[:-2] + "s"
          elif lw.endswith("s") and not lw.endswith("ss"):
              new_w = w + "e"
          else:
              continue
          if w.istitle():
              new_w = new_w[0].upper() + new_w[1:]
          ws[i] = new_w
          return True

      return False

    # Swap two different words randomly
    def word_swap(ws):
      if len(ws) < 2:
          return False

      positions = list(range(len(ws)))
      diff_pairs = [(i, j) for i in positions for j in positions if i < j and ws[i] != ws[j]]

      if not diff_pairs:
          return False

      i, j = random.choice(diff_pairs)
      ws[i], ws[j] = ws[j], ws[i]
      return True

    ops = ["frequent_word_swap", "drop_final_s_es", "swap_suffix", "swap_suffix_words", "plural_swap"]
    for name in random.sample(ops, k=len(ops)):
        if name == "frequent_word_swap" and frequent_word_swap(words):
            break
        elif name == "drop_final_s_es" and drop_final_s_es(words):
            break
        elif name == "swap_suffix" and swap_suffix(words, adj_suffixes):
            break
        elif name == "swap_suffix_words" and swap_suffix_words(words, adj_suffixes):
            break
        elif name == "plural_swap" and plural_swap(words):
            break
    else:
      basic_ops = ["char_swap", "word_swap"]
      for name in random.sample(basic_ops, k=2):
        if name == "char_swap":
          all_pos = [(wi, ci) for wi, w in enumerate(words) for ci in range(len(w))]

          for _ in range(10):
            (wi1, ci1), (wi2, ci2) = random.sample(all_pos, 2)
            if words[wi1][ci1] != words[wi2][ci2]:
                w1, w2 = list(words[wi1]), list(words[wi2])
                w1[ci1], w2[ci2] = w2[ci2], w1[ci1]
                words[wi1], words[wi2] = "".join(w1), "".join(w2)
                break
        elif name == "word_swap" and word_swap(words):
            break


    return " ".join(words).strip()


with open("part2.txt", "w", encoding="utf-8", errors="replace", newline="\n") as w:
      for left in read_pairs_train("/content/train.txt"):
          right = corrupt_sentence(left)
          if right == left: right = left[:-1] + "e."
          right = right.replace("\t", " ")
          w.write(f"{left}\t{right}\n")

## 2. Evaluation

### a)  Check Identical

In [ ]:
# Check for identical left and right texts 
def check_same_pairs(path):
    same_lines = []
    with Path(path).open("r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f, start=1):
            line = line.rstrip("\n").lstrip("\ufeff")
            left, right = line.split("\t", 1)
            if left == right:
                same_lines.append(i)
    return same_lines

same_idx = check_same_pairs("part2.txt")
if same_idx:
    print(f"{len(same_idx)} same pairs")
else:
    print("No identical sentences")

No identical sentences


### b) Beat the Classifier

In [23]:
def get_valid_set(path, valid_lines):
    with Path(path).open("r", encoding="utf-8", errors="replace") as f:
        pairs = [tuple(line.rstrip("\n").lstrip("\ufeff").split("\t", 1)) for line in f]

    split = len(pairs) - valid_lines
    X_valid_texts = [r for _, r in pairs[split:]]

    y_valid = [0] * len(X_valid_texts)

    return X_valid_texts, y_valid

X_valid_texts, y_valid = get_valid_set("/content/train.txt", valid_lines=100_000)
X_valid_texts_new, y_valid_new = get_valid_set("part2.txt", valid_lines=100_000)
print(len(X_valid_texts), len(y_valid))
print(len(X_valid_texts_new), len(y_valid_new))

100000 100000
100000 100000


In [16]:
#original incorrect sentences classification
Xva = tfidf.transform(tqdm(X_valid_texts, unit="doc"))
y_pred = clf.predict(Xva)

acc = accuracy_score(y_valid, y_pred)
print(f"{acc:.4f} ({(y_pred == np.array(y_valid)).sum()}/{len(y_valid)})")

100%|██████████| 100000/100000 [00:24<00:00, 4064.91doc/s]


0.5993 (59930/100000)


In [24]:
#new incorrect sentences classification
Xva = tfidf.transform(tqdm(X_valid_texts_new, unit="doc"))
y_pred = clf.predict(Xva)

acc = accuracy_score(y_valid_new, y_pred)
print(f"{acc:.4f} ({(y_pred == np.array(y_valid_new)).sum()}/{len(y_valid)})")

100%|██████████| 100000/100000 [00:25<00:00, 3926.10doc/s]


0.1848 (18484/100000)
